In [ ]:
import random
from datetime import datetime, timedelta, timezone

import mysql.connector
from faker import Faker
from pyspark.sql import SparkSession, functions as F

In [ ]:
fake = Faker("ko_KR")
random.seed(42)


def make_log(now: datetime, i: int) -> dict:
    run_name = random.choice(["daily_etl", "hourly_etl", "backfill", "ad_hoc"])
    run_id = f"run_{now.strftime('%Y%m%d_%H%M')}_{random.randint(1000, 9999)}"
    entity = random.choice(["job", "task", "table", "metric", "dq"])
    instance = random.choice(["row_count", "error_count", "duration_sec", "null_rate", "freshness_min"])
    name = random.choice(["prod", "stg", "dev", "batch01", "batch02"])
    logical_dt = now - timedelta(minutes=random.randint(0, 24 * 60))
    value = round(random.random() * 1000, 6)
    return {"run_name": run_name, "run_id": run_id, "entity": entity, "instance": instance, "name": name, "value": float(value), "logical_datetime": logical_dt.isoformat()}


def make_dummy_logs(total: int = 10000) -> list[dict]:
    now = datetime.now(timezone.utc)
    base = [make_log(now, i) for i in range(total)]
    half = total // 2
    for i in range(half):
        src = base[i + half]
        base[i]["run_name"] = src["run_name"]
        base[i]["run_id"] = src["run_id"]
        base[i]["entity"] = src["entity"]
        base[i]["instance"] = src["instance"]
        base[i]["name"] = src["name"]
        base[i]["value"] = float(src["value"] + random.random() * 10)
        base[i]["logical_datetime"] = (now - timedelta(minutes=random.randint(0, 60))).isoformat()

    return base

In [ ]:
mysql_host = "mysql-primary"
mysql_port = 3306
mysql_db = "mmix"
mysql_user = "mmix"
mysql_pass = "mmix"
mysql_database = "mmix"
table = "etl_analysis_logs"
driver = "com.mysql.cj.jdbc.Driver"
jdbc_url = f"jdbc:mysql://{mysql_host}:{mysql_port}/{mysql_db}?useUnicode=true&characterEncoding=utf8&useSSL=false&serverTimezone=UTC"
properties = {"user": mysql_user, "password": mysql_pass, "driver": driver}

In [ ]:
spark = SparkSession.builder.appName("Mysql Example").master("spark://localhost:7077").getOrCreate()
dataframe = spark.createDataFrame(make_dummy_logs(100000)).withColumn("logical_datetime", F.to_timestamp("logical_datetime"))
dataframe.write.mode("append").jdbc(url=jdbc_url, table=table, properties=properties)

### MapPartitions & UFD 이용

In [ ]:
batch_size = 500
upsert_sql = f"""
INSERT INTO {table} (run_name, run_id, entity, instance, name, value, logical_datetime) VALUES (%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE value = VALUES(value), logical_datetime = VALUES(logical_datetime), updated_at = CURRENT_TIMESTAMP
"""


def chunked(it, size):
    buf = []
    for x in it:
        buf.append(x)
        if len(buf) >= size:
            yield buf
            buf = []
    if buf:
        yield buf


def upsert_partition(rows_iter):
    conn = mysql.connector.connect(host=mysql_host, port=mysql_port, user=mysql_user, password=mysql_pass, database=mysql_database, autocommit=False)
    cur = conn.cursor()
    processed = 0
    try:
        def to_tuple(r):
            return (r["run_name"], r["run_id"], r["entity"], r["instance"], r["name"], float(r["value"]), r["logical_datetime"])

        for batch in chunked((to_tuple(r.asDict()) for r in rows_iter), batch_size):
            cur.executemany(upsert_sql, batch)
            conn.commit()
            processed += len(batch)
    except Exception:
        conn.rollback()
        raise
    finally:
        try:
            cur.close()
        finally:
            conn.close()

    yield processed

In [ ]:
log_dataframe = spark.createDataFrame(make_dummy_logs(100000)).withColumn("logical_datetime", F.to_timestamp("logical_datetime"))
counts = log_dataframe.repartition(2).rdd.mapPartitions(upsert_partition).collect()

In [ ]:
spark.stop()